# Boxdistractor — MCTS-vs-softFloyd ablation (E1a + E1c)

Chạy trên **Kaggle GPU T4** (env MuJoCo cần stack cũ). Đo robustness của planner khi
world-model bị nhiễu: **E1a** (stochastic) + **E1c** (bias cố định + execution feedback).

**Trước khi chạy:** Settings → Accelerator = **GPU T4**, Internet = **ON**;
**Add Data →** output notebook đã train `boxdistractor_s829` (để restore checkpoint). Eval chỉ cần
`agent.pt`+`algo.pt` (KHÔNG cần replay). BoxDistractor = long-horizon MANIPULATION (analogue của AntMaze bên nav): planner load-bearing → kỳ vọng MCTS ăn tiền dưới nhiễu, giống AntMaze. Train trước bằng boxdistractor_train.ipynb.

## 1. Code + env (~10–15 phút lần đầu)

In [ ]:
import os
if os.path.isdir('/kaggle/working/latent_landmarks'):
    !cd /kaggle/working/latent_landmarks && git pull -q origin retrain
else:
    !git clone -q -b retrain https://github.com/Jun1801/latent_landmarks.git /kaggle/working/latent_landmarks
if not os.path.isdir('/kaggle/working/wmag'):
    !git clone -q https://github.com/LunjunZhang/world-model-as-a-graph /kaggle/working/wmag
!bash /kaggle/working/latent_landmarks/repro/setup_kaggle.sh

## 2. Restore checkpoint (chỉ agent.pt + algo.pt)
Tìm `<SLUG>` bằng `!ls /kaggle/input/` (là dataset/output bạn vừa Add Data).

In [ ]:
!ls /kaggle/input/
import os, shutil
SLUG = 'PUT-DATASET-SLUG-HERE'          # <-- sửa cho khớp /kaggle/input/
CKPT, ENV = 'boxdistractor_s829', 'Box-aside-v0'
src = f'/kaggle/input/{SLUG}/experiments/{ENV}/{CKPT}/state'
dst = f'/kaggle/working/experiments/{ENV}/{CKPT}/state'; os.makedirs(dst, exist_ok=True)
for f in ['agent.pt', 'algo.pt']:
    shutil.copy(f'{src}/{f}', f'{dst}/{f}')
print('restored ->', os.listdir(dst))

## 3. Verify GPU + env

In [ ]:
!export PATH=/opt/conda/bin:$PATH; export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-};  conda run -n l3p python -c "import torch,mujoco_py; print('cuda',torch.cuda.is_available())"

## 4. E1a — σ=0 sanity (mcts ≈ soft_floyd ≈ checkpoint success)

In [ ]:
!export PATH=/opt/conda/bin:$PATH; export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-};  conda run -n l3p python /kaggle/working/latent_landmarks/repro/paper_mcts/eval_ablation.py --env boxdistractor --regime e1a --resume_ckpt boxdistractor_s829 --episodes 2 --n_test_rollouts 20 --sims 100 --sigmas 0

## 5. E1a sweep — phased: classical (fast) → MCTS (slow)

Phase 1 = classical planners (soft_floyd / dijkstra / A\* / greedy), full σ, 3 noise seeds, own
JSON. Phase 2 = MCTS variants (slow). Separate output per phase so a timeout keeps the classical
comparison. BoxDistractor is long-horizon manipulation → planning load-bearing (expect MCTS to win).

In [ ]:
# PHASE 1 — classical planners (fast)
!export PATH=/opt/conda/bin:$PATH; export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-};  conda run -n l3p python /kaggle/working/latent_landmarks/repro/paper_mcts/eval_ablation.py --env boxdistractor --regime e1a --resume_ckpt boxdistractor_s829 --episodes 4 --n_test_rollouts 40 --sims 100 --latency --planners soft_floyd dijkstra astar greedy --noise-seeds 0 1 2 --out /kaggle/working/exp_out/boxdistractor_e1a_classical.json --sigmas 0 5 10 20 40

In [ ]:
# PHASE 2 — MCTS variants (slow)
!export PATH=/opt/conda/bin:$PATH; export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-};  conda run -n l3p python /kaggle/working/latent_landmarks/repro/paper_mcts/eval_ablation.py --env boxdistractor --regime e1a --resume_ckpt boxdistractor_s829 --episodes 3 --n_test_rollouts 30 --sims 100 --planners soft_floyd mcts mcts+suffix mcts+pw mcts+bayes --out /kaggle/working/exp_out/boxdistractor_e1a_mcts.json --sigmas 0 5 10 20 40

## 6. E1c — σ=0 sanity (soft_floyd/mcts_nofb/mcts_fb trùng nhau)

In [ ]:
!export PATH=/opt/conda/bin:$PATH; export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-};  conda run -n l3p python /kaggle/working/latent_landmarks/repro/paper_mcts/eval_ablation.py --env boxdistractor --regime e1c --resume_ckpt boxdistractor_s829 --episodes 2 --n_test_rollouts 20 --sims 100 --sigmas 0

## 7. E1c sweep — phased: classical (fast) → MCTS (slow)

Phase 1 = classical planners (static, no feedback) baseline; Phase 2 = `mcts_nofb` vs `mcts_fb`
(execution feedback). Separate JSON per phase.

In [ ]:
# PHASE 1 — classical planners (fast)
!export PATH=/opt/conda/bin:$PATH; export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-};  conda run -n l3p python /kaggle/working/latent_landmarks/repro/paper_mcts/eval_ablation.py --env boxdistractor --regime e1c --resume_ckpt boxdistractor_s829 --episodes 4 --n_test_rollouts 40 --sims 100 --latency --planners soft_floyd dijkstra astar greedy --noise-seeds 0 1 2 --out /kaggle/working/exp_out/boxdistractor_e1c_classical.json --sigmas 0 0.1 0.2 0.3 0.4 0.5

In [ ]:
# PHASE 2 — MCTS variants (slow)
!export PATH=/opt/conda/bin:$PATH; export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-};  conda run -n l3p python /kaggle/working/latent_landmarks/repro/paper_mcts/eval_ablation.py --env boxdistractor --regime e1c --resume_ckpt boxdistractor_s829 --episodes 3 --n_test_rollouts 30 --sims 100 --planners soft_floyd mcts_nofb mcts_fb --noise-seeds 0 1 2 --out /kaggle/working/exp_out/boxdistractor_e1c_mcts.json --sigmas 0 0.1 0.2 0.3 0.4 0.5

## 8. Dump 3 planner (cho hình plan-comparison + graph wormhole), 1 episode

In [ ]:
!export PATH=/opt/conda/bin:$PATH; export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-};  conda run -n l3p python /kaggle/working/latent_landmarks/repro/paper_mcts/eval_ablation.py --env boxdistractor --regime e1c --resume_ckpt boxdistractor_s829 --episodes 1 --n_test_rollouts 1 --sims 100 --sigmas 0.2 --dump-plans /kaggle/working/exp_out/boxdistractor_plans_e1c.json

## 9. Lấy JSON về (để plot local)
`/kaggle/working/exp_out/` được lưu khi **Save Version**. Tải các file → gửi lại tôi plot:
- `boxdistractor_e1a_classical.json`, `boxdistractor_e1a_mcts.json`
- `boxdistractor_e1c_classical.json`, `boxdistractor_e1c_mcts.json`
- `boxdistractor_plans_e1c.json`

Tôi overlay success-vs-σ tất cả planner (band [min,max] cho classical multi-seed) + hình
wormhole + plan-comparison.

In [ ]:
!ls -la /kaggle/working/exp_out/ 2>/dev/null || echo 'chưa có output'

## Ghi chú
- **σ=0 luôn phải khớp** soft_floyd(clean) — nếu lệch nhiều là port sai, dừng & báo.
- Kết quả in ra bảng success theo σ; copy lại để tổng hợp.
- Chạy dài (nhiều σ × episode × MCTS search) có thể vài chục phút — giảm `--episodes`/`--sims`
  nếu muốn nhanh; `--no_cuda` nếu không có GPU (chậm hơn nhiều).